# Phase 0.1: EOS Verification
## Verify EOS condition λ_max · η_crit ≈ 2

**Protocol:** ConvNet L=5, D=256, BN vs LN, 8 learning rates, 3 seeds, 20 epochs

This notebook verifies the Equation of State (EOS) condition that the product of the maximum singular value at initialization (λ_max) and the critical learning rate (η_crit) should be approximately 2.

**What to expect:**
- λ_max · η ≈ 2 for the critical learning rate region
- Convergence behavior differs between BN and LN

In [ ]:
# Cell 1: Install dependencies
import subprocess
subprocess.run(['pip', 'install', '-q', 'torch', 'torchvision', 'numpy', 'scipy', 'matplotlib', 'pyyaml', 'tqdm'], check=True)
print("Dependencies installed.")

In [ ]:
# Cell 2: Model definitions and utilities (self-contained)
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
import torchvision
import torchvision.transforms as transforms
import numpy as np
from scipy import stats
from scipy.optimize import curve_fit
from typing import Dict, List, Optional, Tuple
import json
import os
from datetime import datetime
from tqdm import tqdm

# =============================================================================
# MODEL DEFINITION
# =============================================================================

class ConvNetL5(nn.Module):
    """ConvNet with L=5 layers, configurable normalization."""
    
    def __init__(self, D: int = 64, num_classes: int = 10, norm_type: str = 'batchnorm',
                 group_size: int = 4, input_channels: int = 3, kernel_size: int = 3, padding: int = 1):
        super().__init__()
        self.D = D
        self.num_classes = num_classes
        self.norm_type = norm_type
        self.L = 5
        self.group_size = group_size
        
        self.conv1 = nn.Conv2d(input_channels, D, kernel_size, padding=padding)
        self.norm1 = self._make_norm(D)
        self.conv2 = nn.Conv2d(D, D, kernel_size, padding=padding)
        self.norm2 = self._make_norm(D)
        self.conv3 = nn.Conv2d(D, D, kernel_size, padding=padding)
        self.norm3 = self._make_norm(D)
        self.conv4 = nn.Conv2d(D, D, kernel_size, padding=padding)
        self.norm4 = self._make_norm(D)
        self.conv5 = nn.Conv2d(D, D, kernel_size, padding=padding)
        self.norm5 = self._make_norm(D)
        self.global_pool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(D, num_classes)
        self.activation = nn.GELU()
        self._initialize_weights()
        self._stored_activations = {}
    
    def _make_norm(self, num_channels: int) -> Optional[nn.Module]:
        if self.norm_type == 'batchnorm':
            return nn.BatchNorm2d(num_channels, affine=False)
        elif self.norm_type == 'layernorm':
            return nn.GroupNorm(1, num_channels, affine=False)
        elif self.norm_type == 'groupnorm':
            return nn.GroupNorm(self.group_size, num_channels, affine=False)
        elif self.norm_type == 'none':
            return None
        else:
            raise ValueError(f"Unknown norm_type: {self.norm_type}")
    
    def _initialize_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, nonlinearity='relu')
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, (nn.BatchNorm2d, nn.GroupNorm)):
                if m.weight is not None:
                    nn.init.ones_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, nonlinearity='relu')
                nn.init.zeros_(m.bias)
    
    def _get_norm_output(self, x: torch.Tensor, norm_layer) -> torch.Tensor:
        if norm_layer is None:
            return x
        if isinstance(norm_layer, nn.BatchNorm2d):
            if self.training:
                return norm_layer(x)
            else:
                return (x - norm_layer.running_mean.view(1, -1, 1, 1)) / \
                       torch.sqrt(norm_layer.running_var.view(1, -1, 1, 1) + norm_layer.eps)
        elif isinstance(norm_layer, (nn.LayerNorm, nn.GroupNorm)):
            return norm_layer(x)
        return x
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.conv1(x)
        x = self.activation(self._get_norm_output(x, self.norm1))
        x = self.conv2(x)
        x = self.activation(self._get_norm_output(x, self.norm2))
        x = self.conv3(x)
        x = self.activation(self._get_norm_output(x, self.norm3))
        x = self.conv4(x)
        x = self.activation(self._get_norm_output(x, self.norm4))
        x = self.conv5(x)
        x = self.activation(self._get_norm_output(x, self.norm5))
        x = self.global_pool(x).flatten(1)
        return self.fc(x)
    
    def get_all_weights(self) -> List[torch.Tensor]:
        return [getattr(self, f'conv{i}').weight.data for i in range(1, 6)]

def create_model(D: int, norm_type: str, **kwargs) -> ConvNetL5:
    return ConvNetL5(D=D, norm_type=norm_type, **kwargs)

# =============================================================================
# ACTIVATION CAPTURE AND SIGMA COMPUTATION
# =============================================================================

def capture_activations(model, dataloader, device):
    """Capture normalized activations per layer."""
    model.eval()
    activations = []
    
    for data, _ in dataloader:
        data = data.to(device)
        x = data
        for i in range(1, 6):
            conv = getattr(model, f'conv{i}')
            norm = getattr(model, f'norm{i}')
            x = conv(x)
            if norm is not None:
                if isinstance(norm, nn.BatchNorm2d):
                    x_norm = (x - norm.running_mean.view(1, -1, 1, 1)) / \
                             torch.sqrt(norm.running_var.view(1, -1, 1, 1) + norm.eps)
                else:
                    x_norm = norm(x)
            else:
                x_norm = x
            activations.append(x_norm.cpu().detach())
            x = model.activation(x_norm if norm is not None else x)
        break
    return activations

def compute_sigma(activations: List[torch.Tensor]) -> np.ndarray:
    """Compute ℓ₂ norm per layer from activations."""
    sigmas = []
    for act in activations:
        act_flat = act.flatten(start_dim=1)
        l2_per_sample = act_flat.norm(p=2, dim=1)
        mean_l2 = l2_per_sample.mean().item()
        sigmas.append(mean_l2)
    return np.array(sigmas)

def compute_gamma_init(sigma_init: np.ndarray, norm_type: str, L: int = 5) -> Optional[float]:
    """
    Compute γ_init = (1/L) * Σ |ln(σ_ref / σ_init)| where σ_ref = 1 for BN/LN/GN.
    
    For normalized types (BN, LN, GN), σ_ref = 1 by definition.
    For 'none', returns None (no natural reference).
    """
    if norm_type in ('batchnorm', 'layernorm', 'groupnorm'):
        sigma_ref = 1.0
        log_ratios = np.abs(np.log(sigma_ref / sigma_init))
        gamma_init = np.mean(log_ratios)
        return float(gamma_init)
    return None

# =============================================================================
# λ_max MEASUREMENT (Power Iteration)
# =============================================================================

def power_iteration_single_layer(W: torch.Tensor, num_iterations: int = 20, tol: float = 1e-6) -> float:
    """Compute λ_max via power iteration."""
    if W.dim() == 4:
        W_mat = W.reshape(W.shape[0], -1)
    else:
        W_mat = W
    
    if W_mat.shape[0] != W_mat.shape[1]:
        M = W_mat.T @ W_mat
    else:
        M = W_mat
    d = M.shape[0]
    
    torch.manual_seed(42)
    b = torch.randn(d)
    b = b / b.norm()
    
    lambda_prev = 0.0
    for _ in range(num_iterations):
        Mb = M @ b
        lambda_curr = Mb.norm().item()
        b = Mb / lambda_curr
        if abs(lambda_curr - lambda_prev) < tol:
            break
        lambda_prev = lambda_curr
    
    return torch.sqrt(torch.tensor(lambda_curr)).item()

def measure_lambda_max_mean(model: nn.Module, device: torch.device, num_iterations: int = 20) -> float:
    """Measure mean λ_max across all layers."""
    model.eval()
    lambda_values = []
    for name, module in model.named_modules():
        if isinstance(module, (nn.Conv2d, nn.Linear)):
            W = module.weight.data.to(device)
            lambda_values.append(power_iteration_single_layer(W, num_iterations))
    return np.mean(lambda_values)

# =============================================================================
# TRAINING UTILITIES
# =============================================================================

def train_single_epoch(model, dataloader, optimizer, device):
    model.train()
    total_loss = 0.0
    for data, target in dataloader:
        data, target = data.to(device), target.to(device)
        optimizer.zero_grad()
        output = model(data)
        loss = F.cross_entropy(output, target)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(dataloader)

@torch.no_grad()
def evaluate(model, dataloader, device):
    model.eval()
    total_loss = 0.0
    for data, target in dataloader:
        data, target = data.to(device), target.to(device)
        output = model(data)
        loss = F.cross_entropy(output, target)
        total_loss += loss.item()
    return total_loss / len(dataloader)

def get_run_id(D: int, norm_type: str, lr: float, seed: int) -> str:
    lr_str = f"{lr:.6f}".rstrip('0')
    return f"norm_{norm_type}_D{D}_lr{lr_str}_seed{seed}"

def save_results(results: Dict, filepath: str):
    os.makedirs(os.path.dirname(filepath) if os.path.dirname(filepath) else '.', exist_ok=True)
    with open(filepath, 'w') as f:
        json.dump(results, f, indent=2)

def save_run_result(result: Dict, output_dir: str):
    os.makedirs(output_dir, exist_ok=True)
    if 'config' in result:
        cfg = result['config']
        run_id = get_run_id(cfg.get('D'), cfg.get('norm_type'), cfg.get('lr'), cfg.get('seed'))
    else:
        run_id = f"run_{hash(str(result))[:16]}"
    filepath = os.path.join(output_dir, f"result_{run_id}.json")
    save_results(result, filepath)

# =============================================================================
# DATA LOADING
# =============================================================================

def load_cifar10(batch_size: int = 128):
    transform_train = transforms.Compose([
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
    ])
    transform_test = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
    ])
    
    # Check for Kaggle paths
    data_dir = '/kaggle/input' if os.path.exists('/kaggle/input') else './data'
    
    trainset = torchvision.datasets.CIFAR10(root=data_dir, train=True, download=True, transform=transform_train)
    testset = torchvision.datasets.CIFAR10(root=data_dir, train=False, download=True, transform=transform_test)
    
    trainloader = DataLoader(trainset, batch_size=batch_size, shuffle=True, num_workers=0)
    testloader = DataLoader(testset, batch_size=batch_size, shuffle=False, num_workers=0)
    
    return trainloader, testloader

print("Model and utilities defined.")

In [ ]:
# Cell 3: Configuration and Execution
OUTPUT_DIR = '/kaggle/working/phase_0.1'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Phase 0.1 Configuration
NORM_TYPES = ['batchnorm', 'layernorm']
D = 256
LR_VALUES = [0.0001, 0.0003, 0.001, 0.003, 0.01, 0.03, 0.1, 0.3]
SEEDS = [42, 43, 44]
EPOCHS = 20
BATCH_SIZE = 128

# Device setup
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Load CIFAR-10
trainloader, testloader = load_cifar10(batch_size=BATCH_SIZE)
print(f"Loaded CIFAR-10: {len(trainloader.dataset)} train, {len(testloader.dataset)} test samples")

# =============================================================================
# TRAINING LOOP
# =============================================================================

eos_results = {}

for norm_type in NORM_TYPES:
    print(f"\n>>> Normalization: {norm_type}")
    eos_results[norm_type] = {}
    
    for lr in LR_VALUES:
        print(f"\n  LR = {lr}")
        eos_results[norm_type][lr] = {}
        
        for seed in SEEDS:
            print(f"    Seed {seed}...", end=" ")
            
            # Check for existing result (resume support)
            run_id = get_run_id(D, norm_type, lr, seed)
            result_path = os.path.join(OUTPUT_DIR, f"result_{run_id}.json")
            if os.path.exists(result_path):
                print(f"[Resume] Found existing result, loading...")
                with open(result_path, 'r') as f:
                    result = json.load(f)
            else:
                # Train
                torch.manual_seed(seed)
                np.random.seed(seed)
                
                model = create_model(D=D, norm_type=norm_type).to(device)
                optimizer = optim.SGD(model.parameters(), lr=lr, momentum=0.9)
                scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
                
                # Measure σ_init at initialization (before training)
                activations_init = capture_activations(model, trainloader, device)
                sigma_init = compute_sigma(activations_init)
                
                # Measure λ_max at initialization
                lambda_max_init = measure_lambda_max_mean(model, device, num_iterations=20)
                
                loss_history = []
                for epoch in tqdm(range(EPOCHS), desc=f"Epochs", leave=False):
                    train_loss = train_single_epoch(model, trainloader, optimizer, device)
                    eval_loss = evaluate(model, testloader, device)
                    loss_history.append(eval_loss)
                    scheduler.step()
                
                # Final measurements
                activations_final = capture_activations(model, trainloader, device)
                sigma_final = compute_sigma(activations_final)
                lambda_max_final = measure_lambda_max_mean(model, device, num_iterations=20)
                
                # Compute γ = (1/L) * Σ |ln(σ_final / σ_init)|
                gamma = float(np.mean(np.abs(np.log(sigma_final / sigma_init))))
                
                # Compute γ_init = (1/L) * Σ |ln(σ_ref / σ_init)| where σ_ref = 1
                gamma_init = compute_gamma_init(sigma_init, norm_type, L=5)
                
                result = {
                    'config': {'D': D, 'norm_type': norm_type, 'lr': lr, 'seed': seed, 'num_epochs': EPOCHS},
                    'sigma_init': sigma_init.tolist(),
                    'sigma_final': sigma_final.tolist(),
                    'gamma': gamma,
                    'gamma_init': gamma_init,
                    'lambda_max_init': float(lambda_max_init),
                    'lambda_max_final': float(lambda_max_final),
                    'loss_history': [float(l) for l in loss_history],
                    'is_converged': loss_history[-1] < loss_history[0] * 0.7,
                    'completed': True,
                }
                
                # Save result
                save_run_result(result, OUTPUT_DIR)
            
            # Store for EOS analysis
            eos_ratio = result['lambda_max_init'] * lr
            eos_results[norm_type][lr][seed] = {
                'lambda_max_init': result['lambda_max_init'],
                'eos_ratio': eos_ratio,
                'is_converged': result.get('is_converged', False),
                'gamma_init': result.get('gamma_init'),
            }
            
            print(f"λ_max={result['lambda_max_init']:.3f}, λ_max*η={eos_ratio:.3f}")

print("\nTraining complete.")

In [ ]:
# Cell 4: Results Analysis and Saving
import matplotlib.pyplot as plt

# EOS Analysis
print("\n" + "="*70)
print("EOS VERIFICATION RESULTS")
print("="*70)

for norm_type in NORM_TYPES:
    print(f"\n{norm_type}:")
    eos_ratios = []
    for lr in LR_VALUES:
        for seed in SEEDS:
            ratio = eos_results[norm_type][lr][seed]['eos_ratio']
            converged = eos_results[norm_type][lr][seed]['is_converged']
            gamma_init = eos_results[norm_type][lr][seed].get('gamma_init')
            eos_ratios.append((lr, seed, ratio, converged, gamma_init))
            
    # Average over seeds
    for lr in LR_VALUES:
        ratios = [r[2] for r in eos_ratios if r[0] == lr]
        converged = [r[3] for r in eos_ratios if r[0] == lr]
        gammas_init = [r[4] for r in eos_ratios if r[0] == lr]
        avg_ratio = np.mean(ratios)
        any_converged = any(converged)
        avg_gamma_init = np.nanmean([g for g in gammas_init if g is not None]) if any(g is not None for g in gammas_init) else None
        print(f"  LR={lr:.4f}: avg λ_max·η={avg_ratio:.3f}, converged={any_converged}, γ_init={avg_gamma_init:.4f if avg_gamma_init else 'N/A'}")

# Generate EOS plot
fig, ax = plt.subplots(figsize=(10, 6))
colors = {'batchnorm': '#E63946', 'layernorm': '#2A9D8F'}
markers = {'batchnorm': 'o', 'layernorm': 's'}

for norm_type in NORM_TYPES:
    lrs = []
    eos_ratios = []
    converged = []
    
    for lr in LR_VALUES:
        for seed in SEEDS:
            lrs.append(lr)
            eos_ratios.append(eos_results[norm_type][lr][seed]['eos_ratio'])
            converged.append(eos_results[norm_type][lr][seed]['is_converged'])
    
    for i in range(len(lrs)):
        ax.scatter(lrs[i], eos_ratios[i], color=colors[norm_type], marker=markers[norm_type],
                  s=60, alpha=0.7 if converged[i] else 0.3,
                  edgecolors='white' if converged[i] else 'none')
    
    ax.scatter([], [], color=colors[norm_type], marker=markers[norm_type], label=norm_type, s=80)

ax.axhline(y=2.0, color='red', linestyle='--', alpha=0.7, label='EOS target (λ_max·η = 2)')
ax.axhspan(1.5, 3.0, alpha=0.1, color='green')
ax.set_xscale('log')
ax.set_xlabel('Learning Rate η', fontsize=12)
ax.set_ylabel('λ_max · η', fontsize=12)
ax.set_title('EOS Verification: λ_max · η', fontsize=14, fontweight='bold')
ax.legend(loc='upper left')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'eos_verification.png'), dpi=150)
plt.show()

# Save final results
save_path = os.path.join(OUTPUT_DIR, 'phase_0.1_results.json')
save_results({'eos_results': eos_results, 'config': {
    'norm_types': NORM_TYPES,
    'D': D,
    'lr_values': LR_VALUES,
    'seeds': SEEDS,
    'epochs': EPOCHS
}}, save_path)
print(f"\nResults saved to {save_path}")